# DxBench runner (Colab / Kaggle, T4)

Runtime: GPU (T4). Run cells top to bottom. Kaggle needs Internet enabled in the notebook settings.

In [ ]:
import os, subprocess

# ---- set these ----
REPO_URL = "https://github.com/vixrino/monarch-fft-mathdx.git"
BRANCH = "main"
CUDA_ARCH = "75"
# MathDx packages, checked on developer.nvidia.com/cufftdx-downloads (2026-11-17).
# 26.x ships for CUDA 13 only; the newest CUDA 12 build is 25.12.1.
MATHDX_URLS = {
    "13": "https://developer.nvidia.com/downloads/compute/cuFFTDx/redist/cuFFTDx/cuda13/nvidia-mathdx-26.10.0-cuda13.tar.gz",
    "12": "https://developer.nvidia.com/downloads/compute/cuFFTDx/redist/cuFFTDx/cuda12/nvidia-mathdx-25.12.1-cuda12.tar.gz",
}
# -------------------

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
cuda_major = nvcc.split("release ")[1].split(".")[0]
MATHDX_URL = MATHDX_URLS[cuda_major]
os.environ.update(WORK=WORK, REPO_URL=REPO_URL, BRANCH=BRANCH, CUDA_ARCH=CUDA_ARCH,
                  MATHDX_URL=MATHDX_URL, REPO=f"{WORK}/dxbench", DX_DIR=f"{WORK}/mathdx")
print(nvcc.strip().splitlines()[-1])
print("work dir:", WORK)
print("MathDx:", MATHDX_URL)

In [ ]:
%%bash
set -e
cmake --version | head -1
# the project and the cuFFTDx examples need CMake >= 3.26
python3 - <<'PY' || pip install -q "cmake>=3.26"
import subprocess, sys
v = subprocess.run(["cmake", "--version"], capture_output=True, text=True).stdout.split()[2]
sys.exit(0 if tuple(map(int, v.split(".")[:2])) >= (3, 26) else 1)
PY
hash -r; cmake --version | head -1

In [ ]:
%%bash
set -e
rm -rf "$REPO"
git clone -q --branch "$BRANCH" "$REPO_URL" "$REPO"
git -C "$REPO" log -1 --oneline

In [ ]:
%%bash
set -e
mkdir -p "$DX_DIR"
cd "$DX_DIR"
f=$(basename "$MATHDX_URL")
[ -f "$f" ] || curl -fL -o "$f" "$MATHDX_URL"
if ! tar -tzf "$f" > /dev/null 2>&1; then
  echo "download is not a tar.gz (license page or login wall?). First bytes:"; head -c 300 "$f"; exit 1
fi
tar -xzf "$f"
# archives unpack to nvidia-mathdx-<ver>/nvidia/mathdx/YY.MM/
ls -d "$DX_DIR"/*/nvidia/mathdx/*/

In [ ]:
import glob
roots = sorted(glob.glob(f"{os.environ['DX_DIR']}/*/nvidia/mathdx/*/"))
assert roots, "no nvidia/mathdx/YY.MM directory after unpacking"
os.environ["MATHDX_ROOT"] = roots[-1].rstrip("/")
print("MATHDX_ROOT =", os.environ["MATHDX_ROOT"])

In [ ]:
%%bash
cd "$REPO" && bash scripts/env_check.sh

In [ ]:
%%bash
set -e
cd "$REPO"
cmake -S . -B build -DCUDA_ARCH=$CUDA_ARCH -DMATHDX_ROOT="$MATHDX_ROOT"
cmake --build build -j
./build/dx_smoke

Smoke test: the official cuFFTDx `convolution_performance` example, built from the examples shipped in the package.

In [ ]:
%%bash
set -e
src=$(find "$MATHDX_ROOT" -name convolution_performance.cu | head -1)
[ -n "$src" ] || { echo "convolution_performance.cu not found in package"; exit 1; }
ex=$(dirname "$(dirname "$src")")
echo "examples dir: $ex"
cmake -S "$ex" -B "$WORK/build_cufftdx_examples" \
  -DCMAKE_BUILD_TYPE=Release \
  -Dmathdx_ROOT="$MATHDX_ROOT" \
  -DCUFFTDX_CUDA_ARCHITECTURES=${CUDA_ARCH}-real
# target name is <dir>_<file> in NVIDIA's examples CMakeLists; read it back to be sure
tgt=$(cmake --build "$WORK/build_cufftdx_examples" --target help | grep -o '[A-Za-z0-9_]*convolution_performance' | head -1)
echo "target: $tgt"
cmake --build "$WORK/build_cufftdx_examples" --target "$tgt" -j
bin=$(find "$WORK/build_cufftdx_examples" -type f -name convolution_performance -perm -u+x | head -1)
echo "running $bin"
"$bin"

## Experiments

Accuracy checks (`ctest`), then one sweep: both binaries and the SASS counts in this session, written to a new
`results/<gpu>/<date>_sweep[_k].jsonl` (never appended to). Figures are drawn from that file only.

In [ ]:
%%bash
cd "$REPO"
ctest --test-dir build --output-on-failure

In [ ]:
%%bash
cd "$REPO"
python3 scripts/sweep.py 2>&1 | tee /tmp/sweep.log

In [ ]:
%%bash
cd "$REPO"
python3 scripts/plot.py --tables

In [ ]:
%%bash
cd "$REPO"
# one 64x64x64 complex fp16 GEMM per variant: 0 plain layouts, 1 suggested layouts, 2 forced MMA tile
./build/mma_probe
bash scripts/sass_check.sh build/mma_probe probe

## Nsight Compute

One launch at the throughput batch per configuration, full section set: the slow Monarch case (fp16 64x64),
the fast one (fp16 16x16) and cuFFTDx fp16 at the same two sizes. About 1 minute per kernel.

In [ ]:
%%bash
cd "$REPO"
# profiles for the current kernels are in results/; set PROFILE=1 to take new ones
if [ "${PROFILE:-0}" = 1 ]; then
bash scripts/profile.sh build/monarch_fft monarch:fp16:4096:64x64 monarch:fp16:256:16x16 2>&1 | grep -v "^==PROF==" | tail -n 20
bash scripts/profile.sh build/cufftdx_block_fft cufftdx:fp16:4096 cufftdx:fp16:256 2>&1 | grep -v "^==PROF==" | tail -n 20
fi

In [ ]:
%%bash
cd "$REPO"
python3 scripts/ncu_summary.py results/*/*_ncu_*.csv | column -t -s $'\t'

## MCP server (phase 5)

End-to-end check of `mcp/server.py` over stdio: every tool on the clean repo, then `diagnose` on three copies
with an injected bug (compile error, wrong twiddle sign, fp16 GEMMs without tensor cores). About 8 minutes,
mostly the four Monarch builds.

In [ ]:
%%bash
cd "$REPO"
pip install -q -r mcp/requirements.txt
python3 -u mcp/smoke_client.py 2>&1 | grep -v "^INFO"

## Agent eval tasks (phase 6)

Validates every task with its reference solution (hidden test, public test, HMMA where required) and records
the reference timings. No model is called.

In [ ]:
%%bash
cd "$REPO"
python3 eval/validate.py

In [ ]:
# results are never edited by hand: download the new files as is (figures are regenerated locally from them)
import glob
new = sorted(glob.glob(f"{os.environ['REPO']}/results/*/*_sweep*.jsonl"), key=os.path.getmtime)[-1:]
new += sorted(glob.glob(f"{os.environ['REPO']}/results/*/*_eval_reference*.jsonl"), key=os.path.getmtime)[-1:]
print("\n".join(new))
try:
    from google.colab import files
    for p in new:
        files.download(p)
except ImportError:
    print("Kaggle: download the files above")